In [4]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [5]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
795,This film fails on every count. For a start it...,negative
928,pardon my spelling. This is probably the funni...,negative
588,ok we have a film that some are calling one of...,negative
273,No movies have grabbed my attention like this ...,positive
363,Doll Master is an example of a lousy horror fi...,negative


In [6]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
795,film fails every count start pretentious striv...,negative
928,pardon spelling probably funniest horror movie...,negative
588,ok film calling one best movie ever but sittin...,negative
273,movie grabbed attention like one ha see wanted...,positive
363,doll master example lousy horror film fallen s...,negative


In [8]:
df['sentiment'].value_counts()

sentiment
negative    264
positive    236
Name: count, dtype: int64

In [9]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
795,film fails every count start pretentious striv...,0
928,pardon spelling probably funniest horror movie...,0
588,ok film calling one best movie ever but sittin...,0
273,movie grabbed attention like one ha see wanted...,1
363,doll master example lousy horror film fallen s...,0


In [11]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [12]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [14]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/mds.shakeer/mlops_sentiments_analysis.mlflow')
dagshub.init(repo_owner='mds.shakeer', repo_name='mlops_sentiments_analysis', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as mds.shakeer

Initialized MLflow to track repo "mds.shakeer/mlops_sentiments_analysis"

Repository mds.shakeer/mlops_sentiments_analysis initialized!

2026/06/28 19:37:57 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ffcb5349ed7845c68bcc2614453bcae8', creation_time=1782655677408, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1782655677408, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [15]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-28 19:37:58,033 - INFO - Starting MLflow run...
2026-06-28 19:37:59,337 - INFO - Logging preprocessing parameters...
2026-06-28 19:38:00,623 - INFO - Initializing Logistic Regression model...
2026-06-28 19:38:00,624 - INFO - Fitting the model...
2026-06-28 19:38:00,703 - INFO - Model training complete.
2026-06-28 19:38:00,708 - INFO - Logging model parameters...
2026-06-28 19:38:01,055 - INFO - Making predictions...
2026-06-28 19:38:01,056 - INFO - Calculating evaluation metrics...
2026-06-28 19:38:01,076 - INFO - Logging evaluation metrics...
2026-06-28 19:38:02,567 - INFO - Saving and logging the model...
2026/06/28 19:38:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-06-28 19:38:43,908 - INFO - Model training and logging completed in 44.57 seconds.
2026-06-28 19:38:43,908 - INFO - Accuracy: 0.608
2026-06-28 19:38:43,908 - INFO - Precision: 0.6140350877192983
2026-06-28 19:38:43,908 - INFO - Recall: 0.5645161290322581
2026-06-28

🏃 View run resilient-tern-317 at: https://dagshub.com/mds.shakeer/mlops_sentiments_analysis.mlflow/#/experiments/0/runs/0c341ef7cd694a8f921652e16b17bff8
🧪 View experiment at: https://dagshub.com/mds.shakeer/mlops_sentiments_analysis.mlflow/#/experiments/0
